# MahjongMaster - treino com interface

Este notebook pode rodar no VSCode/Jupyter local, no Colab com runtime local, ou no Colab remoto.

Importante: um runtime remoto do Colab nao consegue ler automaticamente `C:/Codes/MahjongMaster/dataset` do seu PC. Nesse caso, gere o pacote local com `python scripts/prepare_colab_package.py`, envie o `mahjongmaster_colab_dataset.zip` pelo controle de upload abaixo e clique em **Extrair zip**.


In [ ]:
import sys
from pathlib import Path

print('Python:', sys.version)
try:
    import torch
    print('CUDA disponivel:', torch.cuda.is_available())
    if torch.cuda.is_available():
        for index in range(torch.cuda.device_count()):
            print(f'GPU {index}:', torch.cuda.get_device_name(index))
except Exception as error:
    print('Torch ainda nao carregado:', error)


In [ ]:
import base64
import importlib
import os
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path, PurePosixPath

try:
    widgets = importlib.import_module('ipywidgets')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets'])
    widgets = importlib.import_module('ipywidgets')

ipython_display = importlib.import_module('IPython.display')
HTML = ipython_display.HTML
display = ipython_display.display
clear_output = ipython_display.clear_output

IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}
PACKAGE_NAME = 'mahjongmaster_colab_dataset.zip'
REMOTE_WORK_ROOT = Path('/content/MahjongMaster')
DRIVE_RUNS_DIR = Path('/content/drive/MyDrive/MahjongMaster/runs/detect')
TRAIN_PROCESS = None
TRAIN_THREAD = None
TRAIN_SCRIPT_TEXT = 'from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nfrom ultralytics import RTDETR, YOLO\n\nfrom balance_red_five_dataset import balance_red_fives, format_report\n\n\nROOT = Path(__file__).resolve().parents[1]\nIMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Treina o detector YOLO de pecas.")\n    parser.add_argument("--model", default="yolo11n.pt", help="Checkpoint base do YOLO.")\n    parser.add_argument("--data", default=str(ROOT / "data" / "mahjong_soul.yaml"))\n    parser.add_argument("--epochs", type=int, default=100)\n    parser.add_argument("--imgsz", type=int, default=1600)\n    parser.add_argument("--batch", type=int, default=8)\n    parser.add_argument("--device", default=None, help="Ex.: 0 para GPU, cpu para CPU.")\n    parser.add_argument("--patience", type=int, default=100, help="Epocas sem melhora antes de parar. 0 desativa.")\n    parser.add_argument("--project", default=str(ROOT / "runs" / "detect"))\n    parser.add_argument("--name", default="mahjong_soul_tiles")\n    parser.add_argument("--exist-ok", action="store_true")\n    parser.add_argument("--skip-test", action="store_true", help="Nao roda avaliacao final no split test.")\n    parser.add_argument("--test-split", default="test", help="Split usado para avaliacao final apos o treino.")\n    parser.add_argument("--no-balance-red-fives", action="store_true", help="Desativa oversampling controlado dos 5 vermelhos.")\n    parser.add_argument("--red-five-target-ratio", type=float, default=0.75)\n    parser.add_argument("--red-five-max-multiplier", type=float, default=3.0)\n    parser.add_argument("--red-five-max-copies-per-image", type=int, default=3)\n    parser.add_argument("--red-five-max-new-image-ratio", type=float, default=0.25)\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    if not args.no_balance_red_fives:\n        report = balance_red_fives(\n            Path(args.data),\n            split="train",\n            target_ratio=args.red_five_target_ratio,\n            max_multiplier=args.red_five_max_multiplier,\n            max_copies_per_image=args.red_five_max_copies_per_image,\n            max_new_image_ratio=args.red_five_max_new_image_ratio,\n        )\n        print(format_report(report), flush=True)\n    model_class = RTDETR if "rtdetr" in args.model.lower() else YOLO\n    model = model_class(args.model)\n    model.train(\n        data=args.data,\n        epochs=args.epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        patience=args.patience,\n        project=args.project,\n        name=args.name,\n        exist_ok=args.exist_ok,\n    )\n    run_dir = Path(getattr(getattr(model, "trainer", None), "save_dir", Path(args.project) / args.name))\n    best_path = Path(getattr(getattr(model, "trainer", None), "best", run_dir / "weights" / "best.pt"))\n    if not best_path.exists():\n        best_path = run_dir / "weights" / "best.pt"\n\n    if args.skip_test:\n        print("[TEST] Avaliacao final no split test desativada por --skip-test.", flush=True)\n        return\n    if not best_path.exists():\n        print(f"[TEST] best.pt nao encontrado; teste final ignorado: {best_path}", flush=True)\n        return\n    if not split_has_images(Path(args.data), args.test_split):\n        print(f"[TEST] Split {args.test_split!r} sem imagens locais; teste final ignorado.", flush=True)\n        return\n\n    print(f"[TEST] Avaliando best.pt no split {args.test_split!r}: {best_path}", flush=True)\n    test_model = model_class(str(best_path))\n    test_model.val(\n        data=args.data,\n        split=args.test_split,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        project=str(run_dir),\n        name=args.test_split,\n        exist_ok=True,\n    )\n\n\ndef split_has_images(data_yaml: Path, split: str) -> bool:\n    paths = simple_dataset_yaml_paths(data_yaml)\n    dataset_root = paths.get("path")\n    split_value = paths.get(split)\n    if dataset_root is None or split_value is None:\n        return True\n\n    split_path = split_value if split_value.is_absolute() else dataset_root / split_value\n    if not split_path.exists():\n        return False\n    if split_path.is_file():\n        return True\n    return any(path.suffix.lower() in IMAGE_EXTENSIONS for path in split_path.rglob("*"))\n\n\ndef simple_dataset_yaml_paths(data_yaml: Path) -> dict[str, Path]:\n    if not data_yaml.exists():\n        return {}\n    values: dict[str, Path] = {}\n    for raw_line in data_yaml.read_text(encoding="utf-8").splitlines():\n        line = raw_line.split("#", 1)[0].strip()\n        if ":" not in line:\n            continue\n        key, raw_value = line.split(":", 1)\n        key = key.strip()\n        if key not in {"path", "train", "val", "test"}:\n            continue\n        value = raw_value.strip().strip("\'\\"")\n        if not value or value.startswith("["):\n            continue\n        path = Path(value)\n        if key == "path":\n            values[key] = path if path.is_absolute() else (data_yaml.parent / path).resolve()\n        else:\n            values[key] = path\n    return values\n\n\nif __name__ == "__main__":\n    main()\n'
BALANCE_SCRIPT_TEXT = 'from __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom statistics import median\nfrom typing import Any\n\n\nIMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")\nGENERATED_MARKER = "_red5dup"\nRED_FIVE_CLASSES = ("man_5_red", "pin_5_red", "sou_5_red")\nREGULAR_FIVE_CLASSES = ("man_5", "pin_5", "sou_5")\n\n\n@dataclass\nclass CandidateImage:\n    image_path: Path\n    label_path: Path\n    red_counts: Counter[int]\n\n    @property\n    def distinct_reds(self) -> int:\n        return len(self.red_counts)\n\n    @property\n    def total_reds(self) -> int:\n        return sum(self.red_counts.values())\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Duplica imagens raras com 5 vermelho no split train.")\n    parser.add_argument("--data", default="data/mahjong_soul.yaml", help="YAML do dataset.")\n    parser.add_argument("--split", default="train", help="Split usado para oversampling. Use train.")\n    parser.add_argument("--target-ratio", type=float, default=0.75, help="Alvo relativo a mediana dos 5 normais.")\n    parser.add_argument("--max-multiplier", type=float, default=3.0, help="Limite do alvo vs classe red mais comum.")\n    parser.add_argument("--max-copies-per-image", type=int, default=3)\n    parser.add_argument("--max-new-image-ratio", type=float, default=0.25, help="Maximo de novas imagens vs imagens originais.")\n    parser.add_argument("--dry-run", action="store_true")\n    parser.add_argument("--keep-existing", action="store_true", help="Nao remove copias red5dup antigas antes de balancear.")\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    report = balance_red_fives(\n        Path(args.data),\n        split=args.split,\n        target_ratio=args.target_ratio,\n        max_multiplier=args.max_multiplier,\n        max_copies_per_image=args.max_copies_per_image,\n        max_new_image_ratio=args.max_new_image_ratio,\n        dry_run=args.dry_run,\n        clean_existing=not args.keep_existing,\n    )\n    print(format_report(report), flush=True)\n\n\ndef balance_red_fives(\n    data_yaml: Path,\n    split: str = "train",\n    target_ratio: float = 0.75,\n    max_multiplier: float = 3.0,\n    max_copies_per_image: int = 3,\n    max_new_image_ratio: float = 0.25,\n    dry_run: bool = False,\n    clean_existing: bool = True,\n) -> dict[str, Any]:\n    data_yaml = data_yaml.resolve()\n    dataset_root, names = read_dataset_yaml(data_yaml)\n    class_to_id = {name: class_id for class_id, name in names.items()}\n    missing = [name for name in (*RED_FIVE_CLASSES, *REGULAR_FIVE_CLASSES) if name not in class_to_id]\n    if missing:\n        return {"enabled": False, "reason": f"classes ausentes no YAML: {missing}"}\n\n    image_dir = dataset_root / "images" / split\n    label_dir = dataset_root / "labels" / split\n    if not image_dir.exists() or not label_dir.exists():\n        return {"enabled": False, "reason": f"split {split!r} nao encontrado em {dataset_root}"}\n\n    removed = cleanup_generated_files(image_dir, label_dir) if clean_existing and not dry_run else 0\n    candidates, class_counts, original_image_count = scan_split(image_dir, label_dir, class_to_id)\n    red_ids = [class_to_id[name] for name in RED_FIVE_CLASSES]\n    regular_ids = [class_to_id[name] for name in REGULAR_FIVE_CLASSES]\n    red_counts_before = {names[class_id]: class_counts[class_id] for class_id in red_ids}\n    regular_counts = [class_counts[class_id] for class_id in regular_ids if class_counts[class_id] > 0]\n    if not candidates:\n        return {\n            "enabled": True,\n            "created": 0,\n            "removed_old": removed,\n            "reason": "nenhuma imagem com 5 vermelho no split",\n            "red_counts_before": red_counts_before,\n        }\n\n    base_target = round(median(regular_counts) * max(0.0, target_ratio)) if regular_counts else max(red_counts_before.values())\n    max_current_red = max(red_counts_before.values()) if red_counts_before else 0\n    target = max(max_current_red, min(base_target, round(max_current_red * max(1.0, max_multiplier))))\n    target = max(1, int(target))\n    max_new_images = max(0, round(original_image_count * max(0.0, max_new_image_ratio)))\n\n    copies_by_source: Counter[Path] = Counter()\n    created_files: list[dict[str, str]] = []\n    simulated_counts = Counter(class_counts)\n    created = 0\n\n    ordered_candidates = sorted(\n        candidates,\n        key=lambda item: (-item.distinct_reds, -item.total_reds, item.image_path.name),\n    )\n    while created < max_new_images and any(simulated_counts[class_id] < target for class_id in red_ids):\n        candidate = choose_candidate(ordered_candidates, red_ids, simulated_counts, target, copies_by_source, max_copies_per_image)\n        if candidate is None:\n            break\n        copy_index = copies_by_source[candidate.image_path] + 1\n        copies_by_source[candidate.image_path] = copy_index\n        for class_id, amount in candidate.red_counts.items():\n            simulated_counts[class_id] += amount\n        created += 1\n        if dry_run:\n            continue\n        new_stem = f"{candidate.image_path.stem}{GENERATED_MARKER}{copy_index:02d}"\n        new_image = candidate.image_path.with_name(f"{new_stem}{candidate.image_path.suffix}")\n        new_label = candidate.label_path.with_name(f"{new_stem}.txt")\n        shutil.copy2(candidate.image_path, new_image)\n        shutil.copy2(candidate.label_path, new_label)\n        created_files.append({"image": str(new_image), "label": str(new_label)})\n\n    red_counts_after = {names[class_id]: simulated_counts[class_id] for class_id in red_ids}\n    report = {\n        "enabled": True,\n        "dry_run": dry_run,\n        "split": split,\n        "target": target,\n        "target_ratio": target_ratio,\n        "max_multiplier": max_multiplier,\n        "max_copies_per_image": max_copies_per_image,\n        "max_new_images": max_new_images,\n        "original_images": original_image_count,\n        "candidate_images": len(candidates),\n        "removed_old": removed,\n        "created": created,\n        "red_counts_before": red_counts_before,\n        "red_counts_after": red_counts_after,\n        "regular_five_counts": {names[class_id]: class_counts[class_id] for class_id in regular_ids},\n        "copies_by_source": {path.name: count for path, count in copies_by_source.items()},\n        "created_files": created_files[:20],\n    }\n    if not dry_run:\n        report_path = dataset_root / f"red_five_balance_{split}.json"\n        report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")\n    return report\n\n\ndef choose_candidate(\n    candidates: list[CandidateImage],\n    red_ids: list[int],\n    counts: Counter[int],\n    target: int,\n    copies_by_source: Counter[Path],\n    max_copies_per_image: int,\n) -> CandidateImage | None:\n    deficits = {class_id: max(0, target - counts[class_id]) for class_id in red_ids}\n    best: tuple[float, int, int, str, CandidateImage] | None = None\n    for candidate in candidates:\n        if copies_by_source[candidate.image_path] >= max_copies_per_image:\n            continue\n        gain = sum(min(deficits.get(class_id, 0), amount) for class_id, amount in candidate.red_counts.items())\n        if gain <= 0:\n            continue\n        key = (\n            float(gain),\n            candidate.distinct_reds,\n            candidate.total_reds,\n            "".join(chr(255 - ord(char) % 255) for char in candidate.image_path.name),\n            candidate,\n        )\n        if best is None or key[:4] > best[:4]:\n            best = key\n    return best[4] if best else None\n\n\ndef scan_split(image_dir: Path, label_dir: Path, class_to_id: dict[str, int]) -> tuple[list[CandidateImage], Counter[int], int]:\n    red_ids = {class_to_id[name] for name in RED_FIVE_CLASSES}\n    class_counts: Counter[int] = Counter()\n    candidates: list[CandidateImage] = []\n    image_paths = [\n        path for extension in IMAGE_EXTENSIONS for path in image_dir.glob(f"*{extension}")\n        if GENERATED_MARKER not in path.stem\n    ]\n    for image_path in sorted(image_paths):\n        label_path = label_dir / f"{image_path.stem}.txt"\n        if not label_path.exists():\n            continue\n        ids = label_class_ids(label_path)\n        class_counts.update(ids)\n        red_counts = Counter(class_id for class_id in ids if class_id in red_ids)\n        if red_counts:\n            candidates.append(CandidateImage(image_path, label_path, red_counts))\n    return candidates, class_counts, len(image_paths)\n\n\ndef cleanup_generated_files(image_dir: Path, label_dir: Path) -> int:\n    removed = 0\n    for directory in (image_dir, label_dir):\n        for path in directory.glob(f"*{GENERATED_MARKER}*"):\n            if path.is_file():\n                path.unlink()\n                removed += 1\n    return removed\n\n\ndef label_class_ids(label_path: Path) -> list[int]:\n    ids: list[int] = []\n    for raw_line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():\n        parts = raw_line.split()\n        if not parts:\n            continue\n        try:\n            ids.append(int(float(parts[0])))\n        except ValueError:\n            continue\n    return ids\n\n\ndef read_dataset_yaml(data_yaml: Path) -> tuple[Path, dict[int, str]]:\n    dataset_root = data_yaml.parent\n    names: dict[int, str] = {}\n    in_names = False\n    for raw_line in data_yaml.read_text(encoding="utf-8").splitlines():\n        line = raw_line.split("#", 1)[0].rstrip()\n        stripped = line.strip()\n        if not stripped:\n            continue\n        if stripped == "names:":\n            in_names = True\n            continue\n        if ":" not in stripped:\n            continue\n        key, raw_value = stripped.split(":", 1)\n        key = key.strip()\n        value = raw_value.strip().strip("\'\\"")\n        if key == "path":\n            path = Path(value)\n            dataset_root = path if path.is_absolute() else (data_yaml.parent / path).resolve()\n            in_names = False\n            continue\n        if in_names and key.isdigit():\n            names[int(key)] = value\n    return dataset_root, names\n\n\ndef format_report(report: dict[str, Any]) -> str:\n    if not report.get("enabled"):\n        return f"[BALANCE] desativado: {report.get(\'reason\')}"\n    return (\n        "[BALANCE] red fives | "\n        f"split={report.get(\'split\')} target={report.get(\'target\')} "\n        f"created={report.get(\'created\')}/{report.get(\'max_new_images\')} "\n        f"old_removed={report.get(\'removed_old\')} "\n        f"before={report.get(\'red_counts_before\')} "\n        f"after={report.get(\'red_counts_after\')}"\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
DATA_YAML_TEXT = 'path: ../dataset\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n  0: man_1\n  1: man_2\n  2: man_3\n  3: man_4\n  4: man_5\n  5: man_6\n  6: man_7\n  7: man_8\n  8: man_9\n  9: pin_1\n  10: pin_2\n  11: pin_3\n  12: pin_4\n  13: pin_5\n  14: pin_6\n  15: pin_7\n  16: pin_8\n  17: pin_9\n  18: sou_1\n  19: sou_2\n  20: sou_3\n  21: sou_4\n  22: sou_5\n  23: sou_6\n  24: sou_7\n  25: sou_8\n  26: sou_9\n  27: wind_east\n  28: wind_south\n  29: wind_west\n  30: wind_north\n  31: dragon_white\n  32: dragon_green\n  33: dragon_red\n  34: man_5_red\n  35: pin_5_red\n  36: sou_5_red\n  37: tile_back\n'
REQUIREMENTS_TRAIN_TEXT = '-r requirements.txt\nultralytics>=8.3\nipywidgets>=8\ntensorboard>=2.15\n'


def looks_like_project_root(candidate: Path) -> bool:
    return (candidate / 'scripts' / 'train_yolo.py').exists() and (candidate / 'data' / 'mahjong_soul.yaml').exists()


def ensure_support_files(root: Path) -> None:
    (root / 'scripts').mkdir(parents=True, exist_ok=True)
    (root / 'data').mkdir(parents=True, exist_ok=True)
    train_script = root / 'scripts' / 'train_yolo.py'
    balance_script = root / 'scripts' / 'balance_red_five_dataset.py'
    requirements = root / 'requirements-train.txt'
    data_yaml = root / 'data' / 'mahjong_soul.yaml'
    train_script.write_text(TRAIN_SCRIPT_TEXT, encoding='utf-8')
    balance_script.write_text(BALANCE_SCRIPT_TEXT, encoding='utf-8')
    if not requirements.exists():
        requirements.write_text(REQUIREMENTS_TRAIN_TEXT, encoding='utf-8')
    if not data_yaml.exists():
        data_yaml.write_text(DATA_YAML_TEXT, encoding='utf-8')


def uploaded_relative_path(raw_path: str) -> Path | None:
    parts = [part for part in PurePosixPath(raw_path.replace('\\', '/')).parts if part not in {'', '/'}]
    if not parts or '..' in parts:
        return None
    for index, part in enumerate(parts):
        if part in {'dataset', 'data', 'scripts'} or part == 'requirements-train.txt':
            return Path(*parts[index:])
    if parts[0] in {'images', 'labels'}:
        return Path('dataset', *parts)
    return Path(*parts)


def receive_directory_upload_batch(batch: list[dict[str, str]]) -> str:
    written = 0
    for item in batch:
        relative_path = uploaded_relative_path(str(item.get('path', '')))
        if relative_path is None:
            continue
        target = REMOTE_WORK_ROOT / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(base64.b64decode(item['content']))
        written += 1
    return f'{written} arquivos recebidos'


def finish_directory_upload() -> str:
    ensure_support_files(REMOTE_WORK_ROOT)
    root_input.value = str(REMOTE_WORK_ROOT)
    refresh_status()
    return f'Upload concluido em {REMOTE_WORK_ROOT}'


def register_colab_directory_callbacks() -> bool:
    try:
        colab_output = importlib.import_module('google.colab.output')
    except Exception:
        return False
    colab_output.register_callback('mahjongmaster.receive_directory_upload_batch', receive_directory_upload_batch)
    colab_output.register_callback('mahjongmaster.finish_directory_upload', finish_directory_upload)
    return True


def show_directory_upload(_button=None) -> None:
    if not register_colab_directory_callbacks():
        with output:
            print('[PASTA] Upload de pasta funciona no Colab remoto. Fora dele, use o campo Workspace local ou o upload zip.')
        return
    html = r"""
<div style="font-family: sans-serif; border: 1px solid #cbd5e1; padding: 12px; margin: 8px 0;">
  <p style="margin: 0 0 8px 0;"><b>Upload pasta</b>: escolha a pasta <code>dataset</code> ou a pasta inteira <code>MahjongMaster</code>.</p>
  <input id="mm-folder-input" type="file" webkitdirectory directory multiple />
  <button id="mm-folder-button" style="margin-left: 8px;">Enviar pasta</button>
  <progress id="mm-folder-progress" max="100" value="0" style="width: 260px; margin-left: 8px;"></progress>
  <span id="mm-folder-status" style="margin-left: 8px;"></span>
</div>
<script>
(() => {
  const input = document.getElementById('mm-folder-input');
  const button = document.getElementById('mm-folder-button');
  const progress = document.getElementById('mm-folder-progress');
  const status = document.getElementById('mm-folder-status');

  function arrayBufferToBase64(buffer) {
    const bytes = new Uint8Array(buffer);
    const chunkSize = 0x8000;
    let binary = '';
    for (let i = 0; i < bytes.length; i += chunkSize) {
      binary += String.fromCharCode.apply(null, bytes.subarray(i, i + chunkSize));
    }
    return btoa(binary);
  }

  button.onclick = async () => {
    if (!window.google || !google.colab || !google.colab.kernel) {
      status.textContent = 'Este upload de pasta precisa do Colab.';
      return;
    }
    const files = Array.from(input.files || []);
    if (!files.length) {
      status.textContent = 'Escolha uma pasta primeiro.';
      return;
    }
    button.disabled = true;
    status.textContent = `Enviando ${files.length} arquivos...`;
    const batchSize = 4;
    let batch = [];
    for (let index = 0; index < files.length; index++) {
      const file = files[index];
      const content = arrayBufferToBase64(await file.arrayBuffer());
      batch.push({path: file.webkitRelativePath || file.name, content});
      if (batch.length >= batchSize) {
        await google.colab.kernel.invokeFunction('mahjongmaster.receive_directory_upload_batch', [batch], {});
        batch = [];
      }
      progress.value = Math.round(((index + 1) / files.length) * 100);
      status.textContent = `${index + 1}/${files.length} arquivos`;
    }
    if (batch.length) {
      await google.colab.kernel.invokeFunction('mahjongmaster.receive_directory_upload_batch', [batch], {});
    }
    await google.colab.kernel.invokeFunction('mahjongmaster.finish_directory_upload', [], {});
    status.textContent = 'Upload concluido. Clique em Verificar dataset.';
    button.disabled = false;
  };
})();
</script>
"""
    display(HTML(html))


def find_project_root() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)
    for raw in ('C:/Codes/MahjongMaster', str(REMOTE_WORK_ROOT), '/content'):
        candidates.append(Path(raw))
    for candidate in candidates:
        if looks_like_project_root(candidate):
            return candidate
    return REMOTE_WORK_ROOT if Path('/content').exists() else cwd


def default_project_output(root: Path) -> Path:
    if Path('/content').exists():
        return DRIVE_RUNS_DIR
    return root / 'runs' / 'detect'


def current_project_output() -> Path:
    raw_value = project_output_input.value.strip()
    if raw_value:
        return Path(raw_value).expanduser().resolve()
    return default_project_output(current_root())


def google_drive_mounted() -> bool:
    return (Path('/content/drive/MyDrive').exists() and any(Path('/content/drive/MyDrive').iterdir()))


def path_is_inside_drive(path: Path) -> bool:
    try:
        path.resolve().relative_to(Path('/content/drive').resolve())
        return True
    except ValueError:
        return False


def mount_google_drive(_button=None) -> None:
    try:
        drive = importlib.import_module('google.colab.drive')
    except Exception:
        with output:
            print('[DRIVE] Google Drive so pode ser montado dentro do Colab.')
        return
    with output:
        print('[DRIVE] Montando Google Drive em /content/drive...')
    drive.mount('/content/drive')
    DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
    project_output_input.value = str(DRIVE_RUNS_DIR)
    refresh_status()
    with output:
        print('[DRIVE] Saida dos treinos:', DRIVE_RUNS_DIR)


def count_files(directory: Path, extensions: set[str]) -> int:
    if not directory.exists():
        return 0
    return sum(1 for path in directory.iterdir() if path.suffix.lower() in extensions)


def dataset_summary(root: Path) -> str:
    dataset = root / 'dataset'
    parts = []
    for split in ('train', 'val', 'test'):
        images = count_files(dataset / 'images' / split, IMAGE_EXTENSIONS)
        labels = count_files(dataset / 'labels' / split, {'.txt'})
        parts.append(f'{split}: {images} imgs/{labels} labels')
    return ' | '.join(parts)


def available_devices() -> list[tuple[str, str]]:
    items = [('CPU', 'cpu')]
    try:
        import torch
        if torch.cuda.is_available():
            for index in range(torch.cuda.device_count()):
                items.append((f'GPU {index}: {torch.cuda.get_device_name(index)}', str(index)))
    except Exception:
        pass
    return items


def available_models(root: Path) -> list[tuple[str, str]]:
    defaults = [
        ('yolo11n.pt', 'Nano - mais leve/rapido'),
        ('yolo11s.pt', 'Small - equilibrio inicial'),
        ('yolo11m.pt', 'Medium - mais preciso'),
        ('yolo11l.pt', 'Large - pesado/preciso'),
        ('yolo11x.pt', 'XLarge - mais pesado'),
        ('rtdetr-l.pt', 'RT-DETR Large'),
        ('rtdetr-x.pt', 'RT-DETR XLarge'),
    ]
    seen = {name for name, _description in defaults}
    items = [(f'{name} ({description})', name) for name, description in defaults]
    for path in sorted(root.glob('*.pt')):
        if path.name not in seen:
            items.append((f'{path.name} (arquivo local)', path.name))
    return items


def safe_run_name(text: str) -> str:
    safe = ''.join(char if char.isalnum() or char in ('-', '_') else '_' for char in text).strip('_')
    return safe or 'treino_colab_local'


def uploaded_file_items(upload_widget) -> list[tuple[str, bytes]]:
    value = upload_widget.value
    if isinstance(value, dict):
        return [(name, item['content']) for name, item in value.items()]
    return [(item['name'], item['content']) for item in value]


def save_uploaded_package() -> Path | None:
    items = uploaded_file_items(package_upload)
    if not items:
        return None
    name, content = items[0]
    target = Path('/content') / name if Path('/content').exists() else Path.cwd() / name
    target.write_bytes(content)
    return target


def find_package_file() -> Path | None:
    manual = Path(package_path_input.value).expanduser()
    if package_path_input.value.strip() and manual.exists():
        return manual.resolve()
    for candidate in (
        Path.cwd() / PACKAGE_NAME,
        Path('/content') / PACKAGE_NAME,
        Path('/content/drive/MyDrive/MahjongMaster') / PACKAGE_NAME,
    ):
        if candidate.exists():
            return candidate.resolve()
    return None


def extract_package(package_path: Path, destination: Path = REMOTE_WORK_ROOT) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(package_path) as archive:
        names = [name for name in archive.namelist() if not name.endswith('/')]
        has_flat_workspace = any(name == 'scripts/train_yolo.py' for name in names)
        has_prefixed_workspace = any(name.endswith('/scripts/train_yolo.py') for name in names)
        if has_flat_workspace or not has_prefixed_workspace:
            archive.extractall(destination)
            return destination
        archive.extractall(destination.parent)
        prefix = next(name.split('/', 1)[0] for name in names if name.endswith('/scripts/train_yolo.py'))
        return destination.parent / prefix


project_root = find_project_root()
root_input = widgets.Text(value=str(project_root), description='Workspace', layout=widgets.Layout(width='720px'))
project_output_input = widgets.Text(value=str(default_project_output(project_root)), description='Saida', layout=widgets.Layout(width='720px'))
package_path_input = widgets.Text(value='', description='Zip', placeholder='Opcional: /content/mahjongmaster_colab_dataset.zip', layout=widgets.Layout(width='720px'))
package_upload = widgets.FileUpload(accept='.zip', multiple=False, description='Upload zip')
extract_button = widgets.Button(description='Extrair zip', button_style='warning')
directory_button = widgets.Button(description='Upload pasta', button_style='info')
model_dropdown = widgets.Dropdown(options=available_models(project_root), value='yolo11n.pt', description='Modelo')
epochs_input = widgets.IntText(value=800, description='Epocas')
imgsz_input = widgets.IntText(value=1600, description='Imagem')
batch_input = widgets.IntText(value=5, description='Batch')
patience_input = widgets.IntText(value=0, description='Patience')
device_dropdown = widgets.Dropdown(options=available_devices(), description='Device')
run_name_input = widgets.Text(value='', description='Nome', placeholder='Automatico se vazio', layout=widgets.Layout(width='460px'))
install_button = widgets.Button(description='Instalar deps', button_style='')
refresh_button = widgets.Button(description='Verificar dataset', button_style='info')
start_button = widgets.Button(description='Iniciar treino', button_style='success')
stop_button = widgets.Button(description='Parar', button_style='danger')
tensorboard_button = widgets.Button(description='TensorBoard', button_style='')
drive_button = widgets.Button(description='Montar Drive', button_style='primary')
status_html = widgets.HTML()
output = widgets.Output(layout=widgets.Layout(border='1px solid #334155', max_height='520px', overflow_y='auto'))


def current_root() -> Path:
    return Path(root_input.value).expanduser().resolve()


def default_run_name() -> str:
    return f'{epochs_input.value}e_{Path(model_dropdown.value).stem}_{imgsz_input.value}p_{batch_input.value}b'


def selected_run_name() -> str:
    return safe_run_name(run_name_input.value.strip() or default_run_name())


def build_train_command() -> list[str]:
    root = current_root()
    return [
        sys.executable,
        '-u',
        str(root / 'scripts' / 'train_yolo.py'),
        '--model', str(model_dropdown.value),
        '--data', str(root / 'data' / 'mahjong_soul.yaml'),
        '--epochs', str(int(epochs_input.value)),
        '--imgsz', str(int(imgsz_input.value)),
        '--batch', str(int(batch_input.value)),
        '--patience', str(int(patience_input.value)),
        '--device', str(device_dropdown.value),
        '--project', str(current_project_output()),
        '--name', selected_run_name(),
        '--exist-ok',
    ]


def refresh_status(*_args) -> None:
    root = current_root()
    ok_script = (root / 'scripts' / 'train_yolo.py').exists()
    ok_data = (root / 'data' / 'mahjong_soul.yaml').exists()
    ok_dataset = (root / 'dataset' / 'images' / 'train').exists()
    old_value = model_dropdown.value
    model_dropdown.options = available_models(root)
    values = [value for _label, value in model_dropdown.options]
    model_dropdown.value = old_value if old_value in values else 'yolo11n.pt'
    command = ' '.join(f'"{part}"' if ' ' in part else part for part in build_train_command())
    hint = ''
    if Path('/content').exists() and not ok_script:
        hint = '<br><b>Dica:</b> Colab remoto nao le C:/Codes. Gere o zip local e use Upload zip + Extrair zip.'
    if Path('/content').exists() and path_is_inside_drive(current_project_output()) and not google_drive_mounted():
        hint += '<br><b>Drive:</b> clique em Montar Drive antes de iniciar, senao o best.pt nao fica persistente.'
    status_html.value = (
        f'<b>Workspace:</b> {root}<br>'
        f'<b>Status:</b> script {"OK" if ok_script else "NAO"} | data yaml {"OK" if ok_data else "NAO"} | dataset {"OK" if ok_dataset else "NAO"}<br>'
        f'<b>Dataset:</b> {dataset_summary(root)}<br>'
        f'<b>Run:</b> {selected_run_name()}<br>'
        f'<b>Saida:</b> {current_project_output()}<br>'
        f"<b>best.pt:</b> {current_project_output() / selected_run_name() / 'weights' / 'best.pt'}<br>"
        f'<b>Comando:</b> <code>{command}</code>{hint}'
    )


def extract_uploaded_or_selected_package(_button) -> None:
    uploaded = save_uploaded_package()
    package_path = uploaded or find_package_file()
    with output:
        if package_path is None:
            print('[ZIP] Nenhum zip encontrado. Gere com: python scripts/prepare_colab_package.py')
            print('[ZIP] Depois envie o mahjongmaster_colab_dataset.zip no controle Upload zip.')
            return
        print('[ZIP] Extraindo:', package_path)
        extracted_root = extract_package(package_path)
        print('[ZIP] Workspace extraido em:', extracted_root)
        if not (extracted_root / 'scripts' / 'train_yolo.py').exists():
            print('[ZIP] Aviso: este zip parece antigo e nao inclui scripts/train_yolo.py.')
            print('[ZIP] Gere novamente com a versao atual de scripts/prepare_colab_package.py.')
    root_input.value = str(extracted_root)
    refresh_status()


def install_deps(_button) -> None:
    root = current_root()
    req = root / 'requirements-train.txt'
    command = [sys.executable, '-m', 'pip', 'install']
    if req.exists():
        command.extend(['-r', str(req)])
    else:
        command.extend(['ultralytics', 'tensorboard', 'ipywidgets'])
    with output:
        print('[DEPS] Instalando dependencias...')
        print('[DEPS]', ' '.join(command))
    subprocess.check_call(command)
    with output:
        print('[DEPS] Pronto.')


def stream_training(command: list[str], root: Path) -> None:
    global TRAIN_PROCESS
    start_button.disabled = True
    stop_button.disabled = False
    try:
        with output:
            print('[TRAIN] Workspace:', root)
            print('[TRAIN] Saida:', current_project_output())
            print('[TRAIN] Iniciando:', ' '.join(command))
        TRAIN_PROCESS = subprocess.Popen(
            command,
            cwd=str(root),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert TRAIN_PROCESS.stdout is not None
        for line in TRAIN_PROCESS.stdout:
            with output:
                print(line, end='')
        return_code = TRAIN_PROCESS.wait()
        with output:
            print(f'\n[TRAIN] Finalizado com codigo {return_code}.')
            print('[TRAIN] best.pt esperado em:', current_project_output() / selected_run_name() / 'weights' / 'best.pt')
            print('[TRAIN] teste final esperado em:', current_project_output() / selected_run_name() / 'test')
    finally:
        TRAIN_PROCESS = None
        start_button.disabled = False
        stop_button.disabled = True
        refresh_status()


def start_training(_button) -> None:
    global TRAIN_THREAD
    if TRAIN_PROCESS is not None:
        with output:
            print('[TRAIN] Ja existe um treino rodando.')
        return
    root = current_root()
    if not (root / 'scripts' / 'train_yolo.py').exists():
        with output:
            print('[ERRO] Workspace invalido: scripts/train_yolo.py nao encontrado.')
            print('[ERRO] No Colab remoto, envie e extraia o mahjongmaster_colab_dataset.zip primeiro.')
        return
    if not (root / 'dataset' / 'images' / 'train').exists():
        with output:
            print('[ERRO] Dataset local nao encontrado em dataset/images/train.')
        return
    project_output = current_project_output()
    if Path('/content').exists() and path_is_inside_drive(project_output) and not google_drive_mounted():
        with output:
            print('[ERRO] A Saida esta no Google Drive, mas o Drive ainda nao esta montado.')
            print('[ERRO] Clique em Montar Drive antes de iniciar o treino para preservar o best.pt.')
        return
    project_output.mkdir(parents=True, exist_ok=True)
    command = build_train_command()
    TRAIN_THREAD = threading.Thread(target=stream_training, args=(command, root), daemon=True)
    TRAIN_THREAD.start()


def stop_training(_button) -> None:
    if TRAIN_PROCESS is None:
        return
    with output:
        print('[TRAIN] Parando processo...')
    TRAIN_PROCESS.terminate()


def start_tensorboard(_button) -> None:
    logdir = current_project_output()
    with output:
        print('[TB] Logdir:', logdir)
        print('[TB] Se estiver no VSCode/Jupyter local, abra http://localhost:6006')
    subprocess.Popen([sys.executable, '-m', 'tensorboard.main', '--logdir', str(logdir), '--host', '127.0.0.1', '--port', '6006'])


extract_button.on_click(extract_uploaded_or_selected_package)
drive_button.on_click(mount_google_drive)
directory_button.on_click(show_directory_upload)
install_button.on_click(install_deps)
refresh_button.on_click(lambda button: refresh_status())
start_button.on_click(start_training)
stop_button.on_click(stop_training)
tensorboard_button.on_click(start_tensorboard)
for widget in (root_input, project_output_input, package_path_input, model_dropdown, epochs_input, imgsz_input, batch_input, patience_input, device_dropdown, run_name_input):
    widget.observe(refresh_status, names='value')

stop_button.disabled = True
refresh_status()

controls = widgets.VBox([
    root_input,
    widgets.HBox([project_output_input, drive_button]),
    widgets.HBox([package_upload, extract_button, directory_button]),
    package_path_input,
    widgets.HBox([model_dropdown, device_dropdown]),
    widgets.HBox([epochs_input, imgsz_input, batch_input, patience_input]),
    run_name_input,
    widgets.HBox([refresh_button, install_button, start_button, stop_button, tensorboard_button]),
    status_html,
    output,
])
display(controls)
